# Partie 2 — Étape 1 : Data Preparation

```

## 0. Imports et Configuration

In [ ]:
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
import os
import warnings
warnings.filterwarnings('ignore')

# ── Dossier des données ──────────────────────────────
dossier = r"C:\Users\ibm\OneDrive\Documents\Projet Res Neur\data"


# ── Paramètres ───────────────────────────────────────
CODES_NON_AGRICOLES = set(range(111, 200))
RANDOM_SEED         = 42
NB_TIMESTEPS        = 36  
NB_BANDES_CLIM      = 3   

print('✅ Imports OK')
print(f'📁 Dossier : {dossier}')
print(f'⏱️  Climate : {NB_TIMESTEPS} timesteps × {NB_BANDES_CLIM} bandes = {NB_TIMESTEPS*NB_BANDES_CLIM} bandes total')

✅ Imports OK
📁 Dossier : C:\Users\ibm\OneDrive\Documents\Projet Res Neur\data
⏱️  Climate : 36 timesteps × 3 bandes = 108 bandes total


## 1. Chargement des données Partie 1

In [ ]:
pixels_ar  = np.load(os.path.join(dossier, 'pixels_arkansas.npy'))
labels_ar  = np.load(os.path.join(dossier, 'labels_arkansas.npy'),  allow_pickle=True)
pixels_cal = np.load(os.path.join(dossier, 'pixels_california.npy'))
labels_cal = np.load(os.path.join(dossier, 'labels_california.npy'), allow_pickle=True)

print('✅ Données Partie 1 chargées :')
print(f'   pixels_ar  : {pixels_ar.shape}')   
print(f'   labels_ar  : {labels_ar.shape}')   
print(f'   pixels_cal : {pixels_cal.shape}')  
print(f'   labels_cal : {labels_cal.shape}')  
print(f'\n   Classes AR  : {np.unique(labels_ar)}')
print(f'   Classes CAL : {np.unique(labels_cal)}')

✅ Données Partie 1 chargées :
   pixels_ar  : (10000, 36, 10)
   labels_ar  : (10000,)
   pixels_cal : (10000, 36, 10)
   labels_cal : (10000,)

   Classes AR  : ['Corn' 'Cotton' 'Others' 'Rice' 'Soybeans']
   Classes CAL : ['Alfalfa' 'Almonds' 'Grapes' 'Others' 'Pistachios' 'Rice']


## 2. Fonctions utilitaires

In [ ]:
def resampler_covariable(path_covariable, path_sentinel_ref):
    """
    Charge un GeoTIFF covariable et le reprojette pour correspondre
    exactement à la grille du fichier Sentinel-2 de référence.
    Retourne : array numpy (nb_bandes, H, W) en float32
    """
    with rasterio.open(path_sentinel_ref) as src_ref:
        ref_transform = src_ref.transform
        ref_crs       = src_ref.crs
        ref_height    = src_ref.height
        ref_width     = src_ref.width

    with rasterio.open(path_covariable) as src_cov:
        nb_bandes = src_cov.count
        data_out  = np.zeros((nb_bandes, ref_height, ref_width), dtype=np.float32)
        for b in range(1, nb_bandes + 1):
            reproject(
                source        = rasterio.band(src_cov, b),
                destination   = data_out[b - 1],
                dst_transform = ref_transform,
                dst_crs       = ref_crs,
                resampling    = Resampling.bilinear
            )
    return data_out 


def normaliser_minmax(data):
  
    data_norm = np.zeros_like(data, dtype=np.float32)
    for b in range(data.shape[0]):
        bande      = data[b]
        bmin, bmax = np.nanmin(bande), np.nanmax(bande)
        if bmax > bmin:
            data_norm[b] = (bande - bmin) / (bmax - bmin)
        else:
            data_norm[b] = 0.0
    return data_norm


def resampler_et_restructurer_climate(path_cov, path_sentinel_ref,
                                      nb_timesteps=36, nb_bandes_par_t=3):
    """
    Charge un GeoTIFF climatique temporel et le restructure.

    Le GeoTIFF contient nb_timesteps * nb_bandes_par_t bandes au total,
    organisées comme :
        [temp_t1, precip_t1, dewpoint_t1,
         temp_t2, precip_t2, dewpoint_t2, ...]

    Retourne : (nb_timesteps, nb_bandes_par_t, H, W) en float32, normalisé
    """
    raw = resampler_covariable(path_cov, path_sentinel_ref)
    
    nb_total, H, W = raw.shape
    assert nb_total == nb_timesteps * nb_bandes_par_t, \
        f'Attendu {nb_timesteps*nb_bandes_par_t} bandes, obtenu {nb_total}'

    
    raw_norm = normaliser_minmax(raw)

    
    restructured = raw_norm.reshape(nb_timesteps, nb_bandes_par_t, H, W)
    return restructured 


def mapper_arkansas(c):
    if   c == 1:                     return 'Corn'
    elif c == 2:                     return 'Cotton'
    elif c == 3:                     return 'Rice'
    elif c == 5:                     return 'Soybeans'
    elif c in CODES_NON_AGRICOLES:   return None
    elif c > 0:                      return 'Others'
    else:                            return None


def mapper_california(c):
    if   c == 69:                    return 'Grapes'
    elif c == 36:                    return 'Alfalfa'
    elif c == 3:                     return 'Rice'
    elif c == 75:                    return 'Almonds'
    elif c == 204:                   return 'Pistachios'
    elif c in CODES_NON_AGRICOLES:   return None
    elif c > 0:                      return 'Others'
    else:                            return None


def extraire_cov_statique(covariable_nord, covariable_sud,
                           path_sentinel_nord, path_sentinel_sud,
                           path_cdl_nord, path_cdl_sud,
                           labels_zone, mapper,
                           choix_zone, n_par_classe, seed=42):
    
    np.random.seed(seed)

    def charger_labels_cdl(path_sentinel, path_cdl, mapper_fn):
        with rasterio.open(path_sentinel) as src:
            H, W          = src.height, src.width
            ref_transform = src.transform
            ref_crs       = src.crs
        cdl_arr = np.zeros((H, W), dtype=np.uint8)
        with rasterio.open(path_cdl) as src_cdl:
            reproject(
                source=rasterio.band(src_cdl, 1), destination=cdl_arr,
                dst_transform=ref_transform, dst_crs=ref_crs,
                resampling=Resampling.nearest
            )
        labels_flat = np.array([mapper_fn(int(c)) for c in cdl_arr.flatten()], dtype=object)
        return labels_flat, labels_flat != None, H, W

    labels_n, masque_n, H_n, W_n = charger_labels_cdl(path_sentinel_nord, path_cdl_nord, mapper)
    labels_s, masque_s, H_s, W_s = charger_labels_cdl(path_sentinel_sud,  path_cdl_sud,  mapper)

    nb_b       = covariable_nord.shape[0]
    cov_n_flat = covariable_nord.reshape(nb_b, H_n * W_n).T  
    cov_s_flat = covariable_sud.reshape(nb_b,  H_s * W_s).T

    all_cov = []
    for classe, n_voulu in n_par_classe.items():
        if n_voulu == 0:
            continue
        zone = choix_zone.get(classe, 'Nord')
        cov_flat = cov_n_flat if zone == 'Nord' else cov_s_flat
        lab_flat = labels_n   if zone == 'Nord' else labels_s
        msk_flat = masque_n   if zone == 'Nord' else masque_s

        idx = np.where((lab_flat == classe) & msk_flat)[0]
        idx = idx[idx < len(cov_flat)]
        if len(idx) == 0:
            print(f'  ⚠️  Aucun pixel pour {classe} ({zone})')
            all_cov.append(np.zeros((n_voulu, nb_b), dtype=np.float32))
        elif len(idx) < n_voulu:
            all_cov.append(cov_flat[np.random.choice(idx, n_voulu, replace=True)])
        else:
            all_cov.append(cov_flat[np.random.choice(idx, n_voulu, replace=False)])
        print(f'  ✅ {classe:<14} ({zone:4}) : {n_voulu:,} pixels extraits')

    return np.concatenate(all_cov, axis=0)  


def extraire_cov_temporel(covariable_nord, covariable_sud,
                           path_sentinel_nord, path_sentinel_sud,
                           path_cdl_nord, path_cdl_sud,
                           labels_zone, mapper,
                           choix_zone, n_par_classe, seed=42):
   
    np.random.seed(seed)

    def charger_labels_cdl(path_sentinel, path_cdl, mapper_fn):
        with rasterio.open(path_sentinel) as src:
            H, W          = src.height, src.width
            ref_transform = src.transform
            ref_crs       = src.crs
        cdl_arr = np.zeros((H, W), dtype=np.uint8)
        with rasterio.open(path_cdl) as src_cdl:
            reproject(
                source=rasterio.band(src_cdl, 1), destination=cdl_arr,
                dst_transform=ref_transform, dst_crs=ref_crs,
                resampling=Resampling.nearest
            )
        labels_flat = np.array([mapper_fn(int(c)) for c in cdl_arr.flatten()], dtype=object)
        return labels_flat, labels_flat != None, H, W

    labels_n, masque_n, H_n, W_n = charger_labels_cdl(path_sentinel_nord, path_cdl_nord, mapper)
    labels_s, masque_s, H_s, W_s = charger_labels_cdl(path_sentinel_sud,  path_cdl_sud,  mapper)

    nb_t, nb_b, _, _ = covariable_nord.shape

    cov_n_flat = covariable_nord.reshape(nb_t, nb_b, H_n * W_n).transpose(2, 0, 1)  
    cov_s_flat = covariable_sud.reshape(nb_t, nb_b,  H_s * W_s).transpose(2, 0, 1)

    all_cov = []
    for classe, n_voulu in n_par_classe.items():
        if n_voulu == 0:
            continue
        zone = choix_zone.get(classe, 'Nord')
        cov_flat = cov_n_flat if zone == 'Nord' else cov_s_flat
        lab_flat = labels_n   if zone == 'Nord' else labels_s
        msk_flat = masque_n   if zone == 'Nord' else masque_s

        idx = np.where((lab_flat == classe) & msk_flat)[0]
        idx = idx[idx < len(cov_flat)]
        if len(idx) == 0:
            print(f'  ⚠️  Aucun pixel pour {classe} ({zone})')
            all_cov.append(np.zeros((n_voulu, nb_t, nb_b), dtype=np.float32))
        elif len(idx) < n_voulu:
            all_cov.append(cov_flat[np.random.choice(idx, n_voulu, replace=True)])
        else:
            all_cov.append(cov_flat[np.random.choice(idx, n_voulu, replace=False)])
        print(f'  ✅ {classe:<14} ({zone:4}) : {n_voulu:,} pixels extraits')

    return np.concatenate(all_cov, axis=0) 


print('✅ Fonctions définies')

✅ Fonctions définies


## 3. Resampling des covariables

In [ ]:
p = lambda f: os.path.join(dossier, f)

print('=== Resampling Arkansas ===')

clim_ar_n = resampler_et_restructurer_climate(p('climate_arkansas_north.tif'), p('sentinel2_arkansas_north_2021.tif'))
clim_ar_s = resampler_et_restructurer_climate(p('climate_arkansas_south.tif'), p('sentinel2_arkansas_south_2021.tif'))

soil_ar_n = normaliser_minmax(resampler_covariable(p('soil_arkansas_north.tif'), p('sentinel2_arkansas_north_2021.tif')))
soil_ar_s = normaliser_minmax(resampler_covariable(p('soil_arkansas_south.tif'), p('sentinel2_arkansas_south_2021.tif')))
topo_ar_n = normaliser_minmax(resampler_covariable(p('topo_arkansas_north.tif'), p('sentinel2_arkansas_north_2021.tif')))
topo_ar_s = normaliser_minmax(resampler_covariable(p('topo_arkansas_south.tif'), p('sentinel2_arkansas_south_2021.tif')))

print(f'\n  climate nord : {clim_ar_n.shape} | sud : {clim_ar_s.shape}')  
print(f'  soil    nord : {soil_ar_n.shape} | sud : {soil_ar_s.shape}')    
print(f'  topo    nord : {topo_ar_n.shape} | sud : {topo_ar_s.shape}')    

print('\n=== Resampling California ===')
clim_cal_n = resampler_et_restructurer_climate(p('climate_california_north.tif'), p('sentinel2_california_north_2021.tif'))
clim_cal_s = resampler_et_restructurer_climate(p('climate_california_south.tif'), p('sentinel2_california_south_2021.tif'))
soil_cal_n = normaliser_minmax(resampler_covariable(p('soil_california_north.tif'), p('sentinel2_california_north_2021.tif')))
soil_cal_s = normaliser_minmax(resampler_covariable(p('soil_california_south.tif'), p('sentinel2_california_south_2021.tif')))
topo_cal_n = normaliser_minmax(resampler_covariable(p('topo_california_north.tif'), p('sentinel2_california_north_2021.tif')))
topo_cal_s = normaliser_minmax(resampler_covariable(p('topo_california_south.tif'), p('sentinel2_california_south_2021.tif')))

print(f'\n  climate nord : {clim_cal_n.shape} | sud : {clim_cal_s.shape}')
print(f'  soil    nord : {soil_cal_n.shape} | sud : {soil_cal_s.shape}')
print(f'  topo    nord : {topo_cal_n.shape} | sud : {topo_cal_s.shape}')

print('\n✅ Resampling terminé !')

=== Resampling Arkansas ===

  climate nord : (36, 3, 1486, 1856) | sud : (36, 3, 1486, 1857)
  soil    nord : (3, 1486, 1856) | sud : (3, 1486, 1857)
  topo    nord : (2, 1486, 1856) | sud : (2, 1486, 1857)

=== Resampling California ===

  climate nord : (36, 3, 929, 1114) | sud : (36, 3, 929, 1115)
  soil    nord : (3, 929, 1114) | sud : (3, 929, 1115)
  topo    nord : (2, 929, 1114) | sud : (2, 929, 1115)

✅ Resampling terminé !


## 4. Arkansas — Extraction des covariables

In [5]:
choix_zone_ar = {
    'Soybeans': 'Nord',
    'Rice'    : 'Nord',
    'Corn'    : 'Sud',
    'Cotton'  : 'Sud',
    'Others'  : 'Nord'
}
classes_ar       = ['Soybeans', 'Rice', 'Corn', 'Cotton', 'Others']
n_par_classe_ar  = {c: int((labels_ar == c).sum()) for c in classes_ar}
print('Distribution Arkansas (Partie 1) :')
for c, n in n_par_classe_ar.items():
    print(f'  {c:<14} : {n:,}')
print(f'  Total          : {sum(n_par_classe_ar.values()):,}')

Distribution Arkansas (Partie 1) :
  Soybeans       : 4,272
  Rice           : 2,523
  Corn           : 1,238
  Cotton         : 1,201
  Others         : 766
  Total          : 10,000


In [ ]:
args_ar = dict(
    path_sentinel_nord = p('sentinel2_arkansas_north_2021.tif'),
    path_sentinel_sud  = p('sentinel2_arkansas_south_2021.tif'),
    path_cdl_nord      = p('cdl_arkansas_north_2021.tif'),
    path_cdl_sud       = p('cdl_arkansas_south_2021.tif'),
    labels_zone        = labels_ar,
    mapper             = mapper_arkansas,
    choix_zone         = choix_zone_ar,
    n_par_classe       = n_par_classe_ar
)

print('=== Extraction Climate TEMPOREL — Arkansas ===')
cov_climate_ar = extraire_cov_temporel(clim_ar_n, clim_ar_s, **args_ar)
print(f'\n→ cov_climate_ar : {cov_climate_ar.shape}')  

print('\n=== Extraction Soil — Arkansas ===')
cov_soil_ar = extraire_cov_statique(soil_ar_n, soil_ar_s, **args_ar)
print(f'\n→ cov_soil_ar : {cov_soil_ar.shape}')       

print('\n=== Extraction Topo — Arkansas ===')
cov_topo_ar = extraire_cov_statique(topo_ar_n, topo_ar_s, **args_ar)
print(f'\n→ cov_topo_ar : {cov_topo_ar.shape}')        

=== Extraction Climate TEMPOREL — Arkansas ===
  ✅ Soybeans       (Nord) : 4,272 pixels extraits
  ✅ Rice           (Nord) : 2,523 pixels extraits
  ✅ Corn           (Sud ) : 1,238 pixels extraits
  ✅ Cotton         (Sud ) : 1,201 pixels extraits
  ✅ Others         (Nord) : 766 pixels extraits

→ cov_climate_ar : (10000, 36, 3)

=== Extraction Soil — Arkansas ===
  ✅ Soybeans       (Nord) : 4,272 pixels extraits
  ✅ Rice           (Nord) : 2,523 pixels extraits
  ✅ Corn           (Sud ) : 1,238 pixels extraits
  ✅ Cotton         (Sud ) : 1,201 pixels extraits
  ✅ Others         (Nord) : 766 pixels extraits

→ cov_soil_ar : (10000, 3)

=== Extraction Topo — Arkansas ===
  ✅ Soybeans       (Nord) : 4,272 pixels extraits
  ✅ Rice           (Nord) : 2,523 pixels extraits
  ✅ Corn           (Sud ) : 1,238 pixels extraits
  ✅ Cotton         (Sud ) : 1,201 pixels extraits
  ✅ Others         (Nord) : 766 pixels extraits

→ cov_topo_ar : (10000, 2)


## 5. California — Extraction des covariables

In [ ]:
import rasterio
from rasterio.warp import reproject, Resampling

classes_cal = ['Grapes', 'Rice', 'Alfalfa', 'Almonds', 'Pistachios', 'Others']

def compter_classe_zone(path_sentinel, path_cdl, mapper_fn, classe):
    with rasterio.open(path_sentinel) as src:
        H, W          = src.height, src.width
        ref_transform = src.transform
        ref_crs       = src.crs
    cdl_arr = np.zeros((H, W), dtype=np.uint8)
    with rasterio.open(path_cdl) as src_cdl:
        reproject(
            source=rasterio.band(src_cdl, 1), destination=cdl_arr,
            dst_transform=ref_transform, dst_crs=ref_crs,
            resampling=Resampling.nearest
        )
    labels_flat = np.array([mapper_fn(int(c)) for c in cdl_arr.flatten()], dtype=object)
    return int((labels_flat == classe).sum())

choix_zone_cal   = {}
n_par_classe_cal = {c: int((labels_cal == c).sum()) for c in classes_cal}

for c in classes_cal:
    n_n = compter_classe_zone(p('sentinel2_california_north_2021.tif'), p('cdl_california_north_2021.tif'), mapper_california, c)
    n_s = compter_classe_zone(p('sentinel2_california_south_2021.tif'), p('cdl_california_south_2021.tif'), mapper_california, c)
    choix_zone_cal[c] = 'Nord' if n_n >= n_s else 'Sud'
    print(f'  {c:<14} → Nord:{n_n:,} | Sud:{n_s:,} → choix: {choix_zone_cal[c]}')

print('\nDistribution California (Partie 1) :')
for c, n in n_par_classe_cal.items():
    print(f'  {c:<14} : {n:,}')

In [ ]:
args_cal = dict(
    path_sentinel_nord = p('sentinel2_california_north_2021.tif'),
    path_sentinel_sud  = p('sentinel2_california_south_2021.tif'),
    path_cdl_nord      = p('cdl_california_north_2021.tif'),
    path_cdl_sud       = p('cdl_california_south_2021.tif'),
    labels_zone        = labels_cal,
    mapper             = mapper_california,
    choix_zone         = choix_zone_cal,
    n_par_classe       = n_par_classe_cal
)

print('=== Extraction Climate TEMPOREL — California ===')
cov_climate_cal = extraire_cov_temporel(clim_cal_n, clim_cal_s, **args_cal)
print(f'\n→ cov_climate_cal : {cov_climate_cal.shape}')  

print('\n=== Extraction Soil — California ===')
cov_soil_cal = extraire_cov_statique(soil_cal_n, soil_cal_s, **args_cal)
print(f'\n→ cov_soil_cal : {cov_soil_cal.shape}')       

print('\n=== Extraction Topo — California ===')
cov_topo_cal = extraire_cov_statique(topo_cal_n, topo_cal_s, **args_cal)
print(f'\n→ cov_topo_cal : {cov_topo_cal.shape}')       

=== Extraction Climate TEMPOREL — California ===
  ✅ Grapes         (Sud ) : 2,044 pixels extraits
  ✅ Rice           (Nord) : 1,370 pixels extraits
  ✅ Alfalfa        (Nord) : 500 pixels extraits
  ✅ Almonds        (Sud ) : 2,399 pixels extraits
  ✅ Pistachios     (Sud ) : 564 pixels extraits
  ✅ Others         (Nord) : 3,123 pixels extraits

→ cov_climate_cal : (10000, 36, 3)

=== Extraction Soil — California ===
  ✅ Grapes         (Sud ) : 2,044 pixels extraits
  ✅ Rice           (Nord) : 1,370 pixels extraits
  ✅ Alfalfa        (Nord) : 500 pixels extraits
  ✅ Almonds        (Sud ) : 2,399 pixels extraits
  ✅ Pistachios     (Sud ) : 564 pixels extraits
  ✅ Others         (Nord) : 3,123 pixels extraits

→ cov_soil_cal : (10000, 3)

=== Extraction Topo — California ===
  ✅ Grapes         (Sud ) : 2,044 pixels extraits
  ✅ Rice           (Nord) : 1,370 pixels extraits
  ✅ Alfalfa        (Nord) : 500 pixels extraits
  ✅ Almonds        (Sud ) : 2,399 pixels extraits
  ✅ Pistachios     (

## 6. Vérification et nettoyage NaN

In [ ]:
print('=== Vérification finale ===')
print('\nArkansas :')
print(f'  pixels_ar          : {pixels_ar.shape}')     
print(f'  cov_climate_ar     : {cov_climate_ar.shape}')  
print(f'  cov_soil_ar        : {cov_soil_ar.shape}')     
print(f'  cov_topo_ar        : {cov_topo_ar.shape}')     
print('\nCalifornia :')
print(f'  pixels_cal         : {pixels_cal.shape}')       
print(f'  cov_climate_cal    : {cov_climate_cal.shape}')  
print(f'  cov_soil_cal       : {cov_soil_cal.shape}')  
print(f'  cov_topo_cal       : {cov_topo_cal.shape}')   

print('\nContrôle NaN :')
for nom, arr in [('cov_climate_ar',  cov_climate_ar),
                  ('cov_soil_ar',     cov_soil_ar),
                  ('cov_topo_ar',     cov_topo_ar),
                  ('cov_climate_cal', cov_climate_cal),
                  ('cov_soil_cal',    cov_soil_cal),
                  ('cov_topo_cal',    cov_topo_cal)]:
    nan_count = np.isnan(arr).sum()
    status    = '✅' if nan_count == 0 else f'⚠️  {nan_count} NaN'
    print(f'  {nom:<22} : {status}')

# Remplacer les NaN résiduels par 0
cov_climate_ar  = np.nan_to_num(cov_climate_ar,  nan=0.0)
cov_soil_ar     = np.nan_to_num(cov_soil_ar,     nan=0.0)
cov_topo_ar     = np.nan_to_num(cov_topo_ar,     nan=0.0)
cov_climate_cal = np.nan_to_num(cov_climate_cal, nan=0.0)
cov_soil_cal    = np.nan_to_num(cov_soil_cal,    nan=0.0)
cov_topo_cal    = np.nan_to_num(cov_topo_cal,    nan=0.0)
print('\n✅ NaN remplacés par 0')

## 7. Sauvegarde des fichiers .npy

In [10]:
np.save(os.path.join(dossier, 'cov_climate_arkansas.npy'),   cov_climate_ar)
np.save(os.path.join(dossier, 'cov_soil_arkansas.npy'),      cov_soil_ar)
np.save(os.path.join(dossier, 'cov_topo_arkansas.npy'),      cov_topo_ar)
np.save(os.path.join(dossier, 'cov_climate_california.npy'), cov_climate_cal)
np.save(os.path.join(dossier, 'cov_soil_california.npy'),    cov_soil_cal)
np.save(os.path.join(dossier, 'cov_topo_california.npy'),    cov_topo_cal)

print('✅ Fichiers sauvegardés dans :', dossier)
print()
print('  cov_climate_arkansas.npy   →', cov_climate_ar.shape,  '→ [36 timesteps × (temp, precip, dewpoint)]')
print('  cov_soil_arkansas.npy      →', cov_soil_ar.shape,     '→ [ph, organic_carbon, texture]')
print('  cov_topo_arkansas.npy      →', cov_topo_ar.shape,     '→ [elevation, landforms]')
print('  cov_climate_california.npy →', cov_climate_cal.shape, '→ [36 timesteps × (temp, precip, dewpoint)]')
print('  cov_soil_california.npy    →', cov_soil_cal.shape,    '→ [ph, organic_carbon, texture]')
print('  cov_topo_california.npy    →', cov_topo_cal.shape,    '→ [elevation, landforms]')
print()
print('✅ Data Preparation terminée !')

✅ Fichiers sauvegardés dans : C:\Users\ibm\OneDrive\Documents\Projet Res Neur\data

  cov_climate_arkansas.npy   → (10000, 36, 3) → [36 timesteps × (temp, precip, dewpoint)]
  cov_soil_arkansas.npy      → (10000, 3) → [ph, organic_carbon, texture]
  cov_topo_arkansas.npy      → (10000, 2) → [elevation, landforms]
  cov_climate_california.npy → (10000, 36, 3) → [36 timesteps × (temp, precip, dewpoint)]
  cov_soil_california.npy    → (10000, 3) → [ph, organic_carbon, texture]
  cov_topo_california.npy    → (10000, 2) → [elevation, landforms]

✅ Data Preparation terminée !
